# 垃圾分类 — ResNet18 + ResNet34 迁移学习

本项目采用 **PyTorch** 框架，基于预训练的 **ResNet18** 和 **ResNet34** 进行迁移学习，
替代原有的 MindSpore MobileNetV2 方案。两个模型分别通过两阶段训练（冻结骨干网络 → 全网络微调），
最终以集成推理（ensemble）方式结合两个模型的预测结果，并辅以水平翻转 TTA（Test-Time Augmentation）
来提升分类精度，实现 26 类垃圾的准确识别。

## 1. 环境与导入

In [ ]:
import os
import json
import time
import random
import numpy as np
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset
from torchvision import datasets, transforms, models

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. 训练配置

In [ ]:
# ===== 数据路径 =====
DATA_ROOT = "./datasets/5fbdf571c06d3433df85ac65-momodel/garbage_26x100"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "val")
RESULTS_DIR = "./results"

# ===== 训练超参数 =====
IMAGE_SIZE    = 224
BATCH_SIZE    = 32
NUM_WORKERS   = 4

STAGE1_EPOCHS = 5      # 阶段1: 冻结骨干，只训练 fc
STAGE2_EPOCHS = 20     # 阶段2: 解冻全网络微调

LR_STAGE1     = 1e-3
LR_STAGE2     = 3e-5
WEIGHT_DECAY  = 1e-4

SEED = 42

os.makedirs(RESULTS_DIR, exist_ok=True)
print("Config loaded.")

## 3. 数据加载

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def build_full_train_loader():
    """构建包含 train+val 的完整训练集（数据增强）。"""
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.75, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
            hue=0.05
        ),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.12), ratio=(0.3, 3.3)),
    ])

    train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
    val_dataset   = datasets.ImageFolder(VAL_DIR,   transform=train_transform)

    full_train_dataset = ConcatDataset([train_dataset, val_dataset])

    train_loader = DataLoader(
        full_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    return train_dataset, full_train_dataset, train_loader


set_seed(SEED)
base_train_dataset, full_train_dataset, train_loader = build_full_train_loader()
num_classes = len(base_train_dataset.classes)

print(f"Number of classes: {num_classes}")
print(f"Full train samples: {len(full_train_dataset)}")
print(f"Classes: {base_train_dataset.classes}")

# 保存类别映射
class_to_idx_path = os.path.join(RESULTS_DIR, "class_to_idx.json")
with open(class_to_idx_path, "w", encoding="utf-8") as f:
    json.dump(base_train_dataset.class_to_idx, f, ensure_ascii=False, indent=2)
print("Saved class_to_idx to:", class_to_idx_path)

## 4. 模型定义与训练

In [ ]:
def build_resnet18(num_classes):
    """构建 ResNet18 模型，替换最后一层全连接。"""
    model = models.resnet18(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def build_resnet34(num_classes):
    """构建 ResNet34 模型，替换最后一层全连接。"""
    model = models.resnet34(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def freeze_backbone(model):
    """冻结骨干网络，只保留 fc 层可训练。"""
    for param in model.parameters():
        param.requires_grad = False
    for param in model.fc.parameters():
        param.requires_grad = True


def unfreeze_all(model):
    """解冻全部参数。"""
    for param in model.parameters():
        param.requires_grad = True


def train_one_epoch(model, loader, criterion, optimizer, device):
    """训练一个 epoch，返回 (loss, accuracy)。"""
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, preds = torch.max(outputs, 1)

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        running_correct += torch.sum(preds == labels).item()
        total += batch_size

    epoch_loss = running_loss / total
    epoch_acc  = running_correct / total
    return epoch_loss, epoch_acc


def run_two_stage_training(model, train_loader, device, model_name="resnet"):
    """两阶段训练：Stage1 冻结骨干 → Stage2 全网络微调。"""
    model = model.to(device)

    try:
        criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    except TypeError:
        criterion = nn.CrossEntropyLoss()

    # Stage 1: 只训练 fc
    print(f"\n===== {model_name} Stage 1: train fc only =====")
    freeze_backbone(model)
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_STAGE1, weight_decay=WEIGHT_DECAY
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=STAGE1_EPOCHS)

    for epoch in range(STAGE1_EPOCHS):
        start = time.time()
        loss, acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        scheduler.step()
        print(f"  Stage1 [{epoch+1}/{STAGE1_EPOCHS}] loss={loss:.4f} acc={acc:.4f} time={time.time()-start:.1f}s")

    # Stage 2: 全网络微调
    print(f"\n===== {model_name} Stage 2: fine-tune all =====")
    unfreeze_all(model)
    optimizer = optim.Adam(model.parameters(), lr=LR_STAGE2, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=STAGE2_EPOCHS)

    for epoch in range(STAGE2_EPOCHS):
        start = time.time()
        loss, acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        scheduler.step()
        print(f"  Stage2 [{epoch+1}/{STAGE2_EPOCHS}] loss={loss:.4f} acc={acc:.4f} time={time.time()-start:.1f}s")

    return model


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)

## 5. 执行训练

训练 ResNet18（两阶段：冻结骨干 5 epoch → 全网络微调 20 epoch）。
模型已通过 `train_resnet18.py` 离线训练完成，此处代码仅供参考。

In [ ]:
# ResNet18 训练 (已通过 train_resnet18.py 离线完成)
# 如需重新训练，取消注释以下代码执行

# set_seed(SEED)
# model18 = build_resnet18(num_classes)
# model18 = run_two_stage_training(model18, train_loader, device, model_name="ResNet18")
#
# save_path = os.path.join(RESULTS_DIR, "resnet18_final_full.pth")
# torch.save(model18.state_dict(), save_path)
# print(f"Saved ResNet18 to: {save_path}")

print("ResNet18 训练已离线完成，权重文件: results/resnet18_final_full.pth")

## 6. 执行 ResNet34 训练

训练 ResNet34（两阶段：冻结骨干 5 epoch → 全网络微调 20 epoch）。
模型已通过 `train_resnet34.py` 离线训练完成，此处代码仅供参考。

In [ ]:
# ResNet34 训练 (已通过 train_resnet34.py 离线完成)
# 如需重新训练，取消注释以下代码执行

# set_seed(SEED)
# model34 = build_resnet34(num_classes)
# model34 = run_two_stage_training(model34, train_loader, device, model_name="ResNet34")
#
# save_path = os.path.join(RESULTS_DIR, "resnet34_final_full.pth")
# torch.save(model34.state_dict(), save_path)
# print(f"Saved ResNet34 to: {save_path}")

print("ResNet34 训练已离线完成，权重文件: results/resnet34_final_full.pth")

## 7. 模型推理验证

In [ ]:
# 加载已训练模型，在验证集上简单测试
from torchvision import transforms as T

inverted = {
    0: 'Plastic Bottle', 1: 'Hats', 2: 'Newspaper', 3: 'Cans',
    4: 'Glassware', 5: 'Glass Bottle', 6: 'Cardboard', 7: 'Basketball',
    8: 'Paper', 9: 'Metalware', 10: 'Disposable Chopsticks', 11: 'Lighter',
    12: 'Broom', 13: 'Old Mirror', 14: 'Toothbrush', 15: 'Dirty Cloth',
    16: 'Seashell', 17: 'Ceramic Bowl', 18: 'Paint bucket', 19: 'Battery',
    20: 'Fluorescent lamp', 21: 'Tablet capsules', 22: 'Orange Peel',
    23: 'Vegetable Leaf', 24: 'Eggshell', 25: 'Banana Peel'
}

test_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# 加载 ResNet18
m18 = models.resnet18(pretrained=False)
m18.fc = nn.Linear(m18.fc.in_features, 26)
m18.load_state_dict(torch.load("./results/resnet18_final_full.pth", map_location=device))
m18.to(device).eval()

# 加载 ResNet34
m34 = models.resnet34(pretrained=False)
m34.fc = nn.Linear(m34.fc.in_features, 26)
m34.load_state_dict(torch.load("./results/resnet34_final_full.pth", map_location=device))
m34.to(device).eval()

# 测试几张验证集图片
val_root = "./datasets/5fbdf571c06d3433df85ac65-momodel/garbage_26x100/val/00_00/"
if os.path.isdir(val_root):
    files = os.listdir(val_root)[:5]
    for fname in files:
        img = cv2.imread(os.path.join(val_root, fname))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        tensor = test_transform(img_rgb).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = (m18(tensor) + m34(tensor)) / 2.0
            pred = int(torch.argmax(logits, dim=1).item())
        print(f"{fname} -> {inverted[pred]}")
else:
    print("验证集目录不存在，请检查路径。")

## 8. 作业评分

===========================================
**模型预测代码答题区域**
===========================================

In [ ]:
## 生成 main.py 时请勾选此 cell

import numpy as np
import cv2
import torch
import torch.nn as nn
from torchvision import transforms, models

inverted = {0: 'Plastic Bottle', 1: 'Hats', 2: 'Newspaper', 3: 'Cans', 4: 'Glassware', 5: 'Glass Bottle', 6: 'Cardboard', 7: 'Basketball',
            8: 'Paper', 9: 'Metalware', 10: 'Disposable Chopsticks', 11: 'Lighter', 12: 'Broom', 13: 'Old Mirror', 14: 'Toothbrush',
            15: 'Dirty Cloth', 16: 'Seashell', 17: 'Ceramic Bowl', 18: 'Paint bucket', 19: 'Battery', 20: 'Fluorescent lamp', 21: 'Tablet capsules',
            22: 'Orange Peel', 23: 'Vegetable Leaf', 24: 'Eggshell', 25: 'Banana Peel'}

In [ ]:
## 生成 main.py 时请勾选此 cell

MODEL18_PATH = "./results/resnet18_final_full.pth"
MODEL34_PATH = "./results/resnet34_final_full.pth"
IMAGE_SIZE = 224

_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_model18 = None
_model34 = None

_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])


def _build_resnet18():
    model = models.resnet18(pretrained=False)
    model.fc = nn.Linear(model.fc.in_features, 26)
    return model


def _build_resnet34():
    model = models.resnet34(pretrained=False)
    model.fc = nn.Linear(model.fc.in_features, 26)
    return model


def _load_once():
    global _model18, _model34

    if _model18 is not None and _model34 is not None:
        return

    model18 = _build_resnet18()
    state18 = torch.load(MODEL18_PATH, map_location=_device)
    model18.load_state_dict(state18)
    model18.to(_device)
    model18.eval()

    model34 = _build_resnet34()
    state34 = torch.load(MODEL34_PATH, map_location=_device)
    model34.load_state_dict(state34)
    model34.to(_device)
    model34.eval()

    _model18 = model18
    _model34 = model34


def _prepare(image):
    if not isinstance(image, np.ndarray):
        image = np.array(image)

    if image.ndim != 3:
        raise ValueError("Input image must have shape (H, W, C)")

    if image.shape[2] == 1:
        image = np.repeat(image, 3, axis=2)
    elif image.shape[2] != 3:
        raise ValueError("Input image channel must be 1 or 3")

    if image.dtype != np.uint8:
        image = np.clip(image, 0, 255).astype(np.uint8)

    image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    return image


def predict(image):
    """预测入口：输入 np.ndarray (H, W, C)，返回类别名称字符串。"""
    _load_once()
    image = _prepare(image)

    image_flip = np.ascontiguousarray(image[:, ::-1, :])

    tensor1 = _transform(image).unsqueeze(0).to(_device)
    tensor2 = _transform(image_flip).unsqueeze(0).to(_device)

    with torch.no_grad():
        logits18 = (_model18(tensor1) + _model18(tensor2)) / 2.0
        logits34 = (_model34(tensor1) + _model34(tensor2)) / 2.0
        logits = (logits18 + logits34) / 2.0
        pred_idx = int(torch.argmax(logits, dim=1).item())

    return inverted[pred_idx]

## 9. 本地测试

In [ ]:
image_path = './datasets/5fbdf571c06d3433df85ac65-momodel/garbage_26x100/val/00_00/'
import os
files = os.listdir(image_path)
if files:
    image = cv2.imread(os.path.join(image_path, files[0]))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    result = predict(image)
    print(f"Predicted: {result}")